# `07_bicycle_route_infrastructure_metrics` — Infrastructure Metrics per Municipality
 
This notebook derives `bicycle_route_infrastructure_metrics_by_municipality`, a municipality-level summary of cycling infrastructure type and protection level within the designated bicycle route network. It is one of the inputs to `09_bicycle_route_vs_non_bicycle_route`, where infrastructure metrics are combined with extent metrics, non-route metrics, and spatial/demographic context.
 
The input is `bicycle_route_infrastructure_per_side_per_municipality`, which carries one row per way side per municipality with a `clipped_length_meters` column representing the facility length of that side within the municipality boundary. This notebook aggregates those rows into one row per municipality.
 
---
 
## A note on denominators and what is being measured
 
The metrics in this notebook describe the **composition of cycling infrastructure** within the designated route network — not the extent of the network itself. Extent (total route length, density, per-capita length) is covered in `06_bicycle_route_extent_metrics`. The question here is different: of the designated route network that exists in a municipality, what fraction is physically protected, unprotected, uncertain, or unclassifiable?
 
Two denominator choices arise when computing shares:
 
**All facility km** (`total_facility_km`) — includes `uncertain`, `mixed_traffic_uncertain`, `separate_infrastructure`, `ferry`, and `tagging_conflict` alongside `protected`, `unprotected`, and `no_infrastructure`. This denominator answers: of everything in the designated route network, how much is confirmed protected?
 
**Classifiable facility km** (`classifiable_facility_km`) — includes only `protected`, `unprotected`, and `no_infrastructure` — sides where infrastructure type is known with certainty. This denominator answers: of the network where infrastructure type can be determined, what fraction is protected? It excludes `uncertain` and `mixed_traffic_uncertain` from both numerator and denominator.
 
Both denominators are meaningful and tell different stories. `pct_protected_of_total` is a conservative estimate of protection coverage. `pct_protected_of_classifiable` describes the quality of the described network but is blind to the large share that is unclassifiable. Both are computed and reported. The distinction is the same as `pct_protected_of_infra` used in the thesis analysis.
 
---
 
## Table of Contents
 
1. [Setup](#1-setup)
2. [Compute metrics](#2-compute-metrics)
3. [Result: `bicycle_route_infrastructure_metrics_by_municipality`](#3-result-bicycle_route_infrastructure_metrics_by_municipality)
4. [Descriptive statistics](#4-descriptive-statistics)
5. [Visualisation](#5-visualisation)
---
 

## 1. Setup

In [3]:
import duckdb
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from IPython.utils import io

In [4]:
with io.capture_output() as captured:
    %run /home/name/03_boundaries_population.ipynb

In [5]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipalities_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

## 2. Compute metrics
 
The aggregation pivots `protection_level_per_side` into columns, computes shares under both denominators, and left-joins to the complete municipality table to retain municipalities with zero classified infrastructure.

In [6]:
bicycle_route_infrastructure_metrics_by_municipality = duckdb.sql("""
WITH aggregation AS (
    SELECT
        municipality_code,
        -- Facility km by protection level
        ROUND(SUM(CASE WHEN protection_level_per_side = 'protected'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS protected_facility_km,
        ROUND(SUM(CASE WHEN protection_level_per_side = 'unprotected'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS unprotected_facility_km,
        ROUND(SUM(CASE WHEN protection_level_per_side = 'no_infrastructure'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS no_infrastructure_facility_km,
        ROUND(SUM(CASE WHEN protection_level_per_side = 'separate_infrastructure'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS separate_infrastructure_facility_km,
        ROUND(SUM(CASE WHEN protection_level_per_side = 'uncertain'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS uncertain_facility_km,
        ROUND(SUM(CASE WHEN protection_level_per_side = 'mixed_traffic_uncertain'
                       THEN clipped_length_meters ELSE 0 END) / 1000, 3)
                                                AS mixed_traffic_uncertain_facility_km,
        ROUND(SUM(clipped_length_meters) / 1000, 3)
                                                AS total_facility_km
    FROM bicycle_route_infrastructure_per_side_per_municipality
    GROUP BY municipality_code
),
with_shares AS (
    SELECT *,
        -- Classifiable = protected + unprotected + no_infrastructure
        ROUND(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 3)
                                                AS classifiable_facility_km,
        -- Share of total facility km
        ROUND(protected_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                                                AS pct_protected_of_total,
        ROUND(unprotected_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                                                AS pct_unprotected_of_total,
        ROUND(uncertain_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                                                AS pct_uncertain_of_total,
        ROUND(mixed_traffic_uncertain_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                                                AS pct_mixed_traffic_uncertain_of_total,
        -- Share of classifiable facility km only
        ROUND(protected_facility_km * 100.0 /
              NULLIF(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 0), 2)
                                                AS pct_protected_of_classifiable,
        ROUND(unprotected_facility_km * 100.0 /
              NULLIF(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 0), 2)
                                                AS pct_unprotected_of_classifiable
    FROM aggregation
)
SELECT
    m.municipality_code,
    m.municipality_name,
    m.geometry,
    m.area_km2,
    m.population,
    COALESCE(a.protected_facility_km,                  0) AS protected_facility_km,
    COALESCE(a.unprotected_facility_km,                0) AS unprotected_facility_km,
    COALESCE(a.no_infrastructure_facility_km,          0) AS no_infrastructure_facility_km,
    COALESCE(a.separate_infrastructure_facility_km,    0) AS separate_infrastructure_facility_km,
    COALESCE(a.uncertain_facility_km,                  0) AS uncertain_facility_km,
    COALESCE(a.mixed_traffic_uncertain_facility_km,    0) AS mixed_traffic_uncertain_facility_km,
    COALESCE(a.total_facility_km,                      0) AS total_facility_km,
    COALESCE(a.classifiable_facility_km,               0) AS classifiable_facility_km,
    a.pct_protected_of_total,
    a.pct_unprotected_of_total,
    a.pct_uncertain_of_total,
    a.pct_mixed_traffic_uncertain_of_total,
    a.pct_protected_of_classifiable,
    a.pct_unprotected_of_classifiable
FROM municipalities_arrow m
LEFT JOIN with_shares a
    ON m.municipality_code = a.municipality_code
""")
 
bicycle_route_infrastructure_metrics_by_municipality_gdf = gpd.GeoDataFrame.from_arrow(
    bicycle_route_infrastructure_metrics_by_municipality.arrow()
)

CatalogException: Catalog Error: Table with name bicycle_route_infrastructure_per_side_per_municipality does not exist!
Did you mean "pg_prepared_statements"?

 
---
 
## 3. Result: `bicycle_route_infrastructure_metrics_by_municipality`
 


In [ ]:
# Row count — should equal total number of municipalities (342)
print(f"Municipalities: {len(bicycle_route_infrastructure_metrics_by_municipality_gdf):,}")
 
# National totals
duckdb.sql("""
SELECT
    ROUND(SUM(protected_facility_km),               1) AS protected_facility_km,
    ROUND(SUM(unprotected_facility_km),             1) AS unprotected_facility_km,
    ROUND(SUM(no_infrastructure_facility_km),       1) AS no_infrastructure_facility_km,
    ROUND(SUM(uncertain_facility_km),               1) AS uncertain_facility_km,
    ROUND(SUM(mixed_traffic_uncertain_facility_km), 1) AS mixed_traffic_uncertain_facility_km,
    ROUND(SUM(total_facility_km),                   1) AS total_facility_km,
    ROUND(SUM(classifiable_facility_km),            1) AS classifiable_facility_km,
    ROUND(SUM(protected_facility_km) * 100.0 /
          NULLIF(SUM(total_facility_km), 0), 2)         AS pct_protected_of_total,
    ROUND(SUM(protected_facility_km) * 100.0 /
          NULLIF(SUM(classifiable_facility_km), 0), 2)  AS pct_protected_of_classifiable
FROM bicycle_route_infrastructure_metrics_by_municipality
""")

## 4. Descriptive statistics

In [ ]:
infra_metrics = [
    'pct_protected_of_total',
    'pct_protected_of_classifiable',
    'pct_uncertain_of_total',
    'pct_mixed_traffic_uncertain_of_total'
]
 
desc = bicycle_route_infrastructure_metrics_by_municipality_gdf[infra_metrics].describe().T
display(desc.round(2))

`pct_protected_of_total` and `pct_protected_of_classifiable` will differ substantially for municipalities where `mixed_traffic_uncertain` accounts for a large share of the network. A municipality with a high `pct_protected_of_classifiable` but a low `pct_protected_of_total` is one where much of the designated route network lacks cycleway tags — the described portion is predominantly protected, but the tagging coverage is incomplete. Conversely, a municipality with similar values on both metrics has high tagging completeness.
 
`pct_uncertain_of_total` and `pct_mixed_traffic_uncertain_of_total` together characterise OSM tagging completeness for the municipality's designated route network. High values on either indicate that infrastructure type cannot be determined for a large share of the network from OSM data alone.
 

---
 
## 5. Visualisation

Comparing `pct_protected_of_total` (top left) against `pct_mixed_traffic_uncertain_of_total` (bottom left) reveals the tagging completeness gradient. Municipalities where both are high are those where the designated route network is genuinely well-mapped but heavily mixed-traffic; municipalities where `pct_protected_of_total` is low but `pct_mixed_traffic_uncertain_of_total` is high may simply have incomplete tagging rather than genuinely unprotected infrastructure.